# Optiver Submission Notebook (Saved Model Inference)

This notebook loads your trained Vision Transformer model, runs inference on Kaggle test data, and writes `submission.csv` for the Optiver Realized Volatility Prediction competition.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from tqdm.auto import tqdm

In [ ]:
# Environment-aware defaults (Kaggle or Colab/local).
IS_KAGGLE = Path('/kaggle/input').exists()

if IS_KAGGLE:
    DATA_DIR = Path('/kaggle/input/optiver-realized-volatility-prediction')
    OUTPUT_DIR = Path('/kaggle/working')
    # Update this to your uploaded model path in Kaggle Input.
    MODEL_PATH = Path('/kaggle/input/your-model-dataset/best_model.pth')
else:
    DATA_DIR = Path('/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/kaggle_data/optiver-realized-volatility-prediction')
    OUTPUT_DIR = Path('/content')
    MODEL_PATH = Path('/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/best_model.pth')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'MODEL_PATH: {MODEL_PATH}')

In [ ]:
class OrderFlowRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model(
            'vit_base_patch16_224',
            pretrained=False,
            in_chans=4,
            num_classes=1,
            dynamic_img_size=True
        )

    def forward(self, x):
        h, w = x.shape[2], x.shape[3]
        pad_h = (16 - h % 16) % 16
        pad_w = (16 - w % 16) % 16
        x = F.pad(x, (0, pad_w, 0, pad_h))
        return self.model(x).squeeze(-1)

In [ ]:
def encode_to_image(book: pd.DataFrame, trade: pd.DataFrame, output_size=(600, 80, 4), pad=True):
    n_time, n_price, n_channels = output_size
    image = np.zeros((n_time, n_price, n_channels), dtype=np.int32)

    if book.empty:
        return image

    min_price = min(book['bid_price1'].min(), book['bid_price2'].min(), book['ask_price1'].min(), book['ask_price2'].min())
    max_price = max(book['bid_price1'].max(), book['bid_price2'].max(), book['ask_price1'].max(), book['ask_price2'].max())

    if not trade.empty:
        min_price = min(min_price, trade['price'].min())
        max_price = max(max_price, trade['price'].max())

    padding = 0.001 * (max_price - min_price + 1e-8)
    bounds = (min_price - padding, max_price + padding)
    price_edges = np.linspace(bounds[0], bounds[1], n_price + 1)

    def _safe_bins(values):
        bins = np.searchsorted(price_edges, values, side='right') - 1
        return np.clip(bins, 0, n_price - 1)

    sec = book['seconds_in_bucket'].to_numpy(dtype=np.int32)
    valid_sec = (sec >= 0) & (sec < n_time)

    bid_bin_1 = _safe_bins(book['bid_price1'].to_numpy())
    bid_bin_2 = _safe_bins(book['bid_price2'].to_numpy())
    ask_bin_1 = _safe_bins(book['ask_price1'].to_numpy())
    ask_bin_2 = _safe_bins(book['ask_price2'].to_numpy())

    bs1 = book['bid_size1'].to_numpy(dtype=np.int32)
    bs2 = book['bid_size2'].to_numpy(dtype=np.int32)
    as1 = book['ask_size1'].to_numpy(dtype=np.int32)
    as2 = book['ask_size2'].to_numpy(dtype=np.int32)

    np.add.at(image, (sec[valid_sec], bid_bin_1[valid_sec], 0), bs1[valid_sec])
    np.add.at(image, (sec[valid_sec], bid_bin_2[valid_sec], 0), bs2[valid_sec])
    np.add.at(image, (sec[valid_sec], ask_bin_1[valid_sec], 1), as1[valid_sec])
    np.add.at(image, (sec[valid_sec], ask_bin_2[valid_sec], 1), as2[valid_sec])

    if not trade.empty:
        trade_sec = trade['seconds_in_bucket'].to_numpy(dtype=np.int32)
        valid_trade_sec = (trade_sec >= 0) & (trade_sec < n_time)
        trade_bin = _safe_bins(trade['price'].to_numpy())
        trade_size = trade['size'].to_numpy(dtype=np.int32)
        trade_oc = trade['order_count'].to_numpy(dtype=np.int32)

        np.add.at(image[:, :, 2], (trade_sec[valid_trade_sec], trade_bin[valid_trade_sec]), trade_size[valid_trade_sec])

        if pad:
            left_mask = valid_trade_sec & (trade_bin - 1 >= 0)
            right_mask = valid_trade_sec & (trade_bin + 1 < n_price)
            np.add.at(image[:, :, 2], (trade_sec[left_mask], trade_bin[left_mask] - 1), trade_size[left_mask])
            np.add.at(image[:, :, 2], (trade_sec[right_mask], trade_bin[right_mask] + 1), trade_size[right_mask])

        np.add.at(image[:, :, 3], (trade_sec[valid_trade_sec], trade_bin[valid_trade_sec]), trade_oc[valid_trade_sec])

    return image

In [ ]:
model = OrderFlowRegressor().to(device)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model file not found: {MODEL_PATH}')

state = torch.load(MODEL_PATH, map_location=device)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']

model.load_state_dict(state)
model.eval()
print('Model loaded successfully.')

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')
preds = np.zeros(len(test_df), dtype=np.float32)

book_root = DATA_DIR / 'book_test.parquet'
trade_root = DATA_DIR / 'trade_test.parquet'

batch_size = 128

for stock_id, group_idx in tqdm(test_df.groupby('stock_id').groups.items(), desc='Stocks'):
    stock_indices = np.array(list(group_idx), dtype=np.int64)
    stock_time_ids = test_df.loc[stock_indices, 'time_id'].to_numpy()

    stock_book_path = book_root / f'stock_id={stock_id}'
    stock_trade_path = trade_root / f'stock_id={stock_id}'

    book_stock = pd.read_parquet(stock_book_path) if stock_book_path.exists() else pd.DataFrame()
    trade_stock = pd.read_parquet(stock_trade_path) if stock_trade_path.exists() else pd.DataFrame()

    book_groups = {k: v for k, v in book_stock.groupby('time_id')} if not book_stock.empty else {}
    trade_groups = {k: v for k, v in trade_stock.groupby('time_id')} if not trade_stock.empty else {}

    images = []
    for time_id in stock_time_ids:
        book_slice = book_groups.get(time_id, pd.DataFrame())
        trade_slice = trade_groups.get(time_id, pd.DataFrame())
        image = encode_to_image(book_slice, trade_slice, output_size=(600, 80, 4), pad=True)
        images.append(image)

    for start in range(0, len(images), batch_size):
        end = min(start + batch_size, len(images))
        batch_np = np.stack(images[start:end], axis=0)
        batch_tensor = torch.from_numpy(batch_np).permute(0, 3, 1, 2).to(device=device, dtype=torch.float32)

        with torch.no_grad():
            batch_pred = model(batch_tensor).detach().cpu().numpy().astype(np.float32)

        preds[stock_indices[start:end]] = batch_pred

submission = test_df[['row_id']].copy()
submission['target'] = preds

submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved submission to: {submission_path}')
submission.head()

## Submit on Kaggle
If you are in a Kaggle notebook, run:

`!kaggle competitions submit -c optiver-realized-volatility-prediction -f /kaggle/working/submission.csv -m "ViT inference from saved model"`